In [ ]:
print('========= THE ANALYSIS TABLE - standard  DUCKDB ')

In [ ]:
# ------------- configure dataframe printing layout 
import pandas as pd

pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
#------------------------------------------------

In [ ]:
# IMPORTS  single parquet file  TO DUCKDB

import numpy as np
import duckdb
from pathlib import Path
from binance.client import Client
from datetime import datetime
import time
import plotly.graph_objects as go 
import plotly.express as px



# single file path

# pqt_file = Path.cwd()/'__inputs'/'sample_aggt_pqts'/'SOLUSDT-aggTrades-2026-07-07.parquet'

folder = Path.cwd()/'__inputs'/'sample_aggt_pqts'
sorted_pqt_files = sorted(folder.glob('*parquet'))
pqt_file = [pqt.as_posix() for pqt in sorted_pqt_files]

# symbol name
# SYMBOL = pqt_file.name.split("-")[0]
SYMBOL = "SOLUSDT"

# chart naming
fname = datetime.now().strftime("%Y%m%d_%H%M%S")
stt =datetime.now() # start time

print(SYMBOL)
# print(pqt_file)
print(folder.exists())


In [ ]:
# LOAD  aggregated DATA TO dataframe with  DUCKDB

conn = duckdb.connect(database=":memory:")

qry = f"""--sql
SELECT
    time_bucket(INTERVAL '5 Minutes', epoch_ms(CAST(transact_time AS BIGINT))) AS transact_time,
    arg_min(price, transact_time) AS tde1_price,
    arg_max(price, transact_time) AS last_price,
    sum(CASE WHEN is_buyer_maker = FALSE THEN (price * quantity) ELSE 0.0 END) AS buyVol_usdt,
    sum(CASE WHEN is_buyer_maker = TRUE  THEN (price * quantity) ELSE 0.0 END) AS sellVol_usdt
FROM read_parquet({pqt_file})
GROUP BY 1
ORDER BY 1 ASC;
"""

bdf = conn.execute(qry).df()
conn.close()

print(bdf.tail())